In [1]:
import io
import pandas as pd
from pathlib import Path

RAW, PROC = Path("../data/raw"), Path("../data/processed")
REPORT_DATE = pd.Timestamp("2026-11-06")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

def read_tga(fn, source):
    text = (RAW / fn).read_text(encoding="utf-8-sig", errors="replace")
    pos = text.index('\n"ARTG ID"') + 1
    df = pd.read_csv(io.StringIO(text[pos:]), dtype=str, index_col=False)
    df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
    df["source"] = source
    return df

tga = pd.concat([read_tga("tga_shortages_active_20261106.csv", "active"),
                 read_tga("tga_shortages_archived_20261106.csv", "archived")], ignore_index=True)
print("rows by source:\n", tga["source"].value_counts().to_string())

for c in ["supply_impact_start_date", "supply_impact_end_date", "deletion_from_market", "last_updated"]:
    parsed = pd.to_datetime(tga[c], format="%d/%m/%Y", errors="coerce")
    print(c, "| unparsed:", (parsed.isna() & tga[c].notna()).sum())
    tga[c] = parsed

tga = tga.drop(columns=["phone"])
for c in ["artg_id", "artg_name", "active_ingredients", "dosage_form", "sponsor", "shortage_status",
          "shortage_impact_rating", "reason", "availability", "management_action"]:
    tga[c] = tga[c].str.strip()

print("\nmissing artg_name / active_ingredients:")
print(tga[tga["artg_name"].isna() | tga["active_ingredients"].isna()][["artg_id", "artg_name", "active_ingredients", "source"]].to_string())

tga["has_start"] = tga["supply_impact_start_date"].notna()
tga["has_end"] = tga["supply_impact_end_date"].notna()
tga["has_deletion"] = tga["deletion_from_market"].notna()
print("\nsource x status x has_start/has_end/has_deletion:")
print(tga.groupby(["source", tga["shortage_status"].fillna("(none)"), "has_start", "has_end", "has_deletion"]).size().to_string())

print("\nend < start:", (tga["supply_impact_end_date"] < tga["supply_impact_start_date"]).groupby(tga["source"]).sum().to_dict())
print("end > report date:", (tga["supply_impact_end_date"] > REPORT_DATE).groupby(tga["source"]).sum().to_dict())
print("start > report date:", (tga["supply_impact_start_date"] > REPORT_DATE).groupby(tga["source"]).sum().to_dict())

key = ["artg_id", "supply_impact_start_date"]
print("\nrows sharing (artg_id, start) within same source:", len(tga[tga.duplicated(key + ["source"], keep=False) & tga["has_start"]]))
a_res = tga[(tga["source"] == "active") & (tga["shortage_status"] == "Resolved")]
m = a_res.merge(tga[tga["source"] == "archived"][key].drop_duplicates(), on=key, how="inner")
print("active Resolved:", len(a_res), "| also in archived by (artg_id, start):", len(m))

rows by source:
 source
archived    9697
active      1044
supply_impact_start_date | unparsed: 0
supply_impact_end_date | unparsed: 0
deletion_from_market | unparsed: 0
last_updated | unparsed: 0

missing artg_name / active_ingredients:
     artg_id               artg_name                        active_ingredients    source
5592  221118                     NaN  hydrochlorothiazide~olmesartan medoxomil  archived
7816   32486  ALBUMIN-SALINE Diluent                                       NaN  archived

source x status x has_start/has_end/has_deletion:
source    shortage_status  has_start  has_end  has_deletion
active    Anticipated      True       True     False             91
          Current          True       True     False            448
          Discontinued     False      False    True             292
          Resolved         True       True     False            212
                                               True               1
archived  (none)           True       False  

In [2]:
cols = ["artg_id", "artg_name", "source", "shortage_status", "supply_impact_start_date",
        "supply_impact_end_date", "deletion_from_market", "shortage_impact_rating", "reason"]
def show(df, n=None):
    out = df[cols].assign(artg_name=df["artg_name"].str[:28], reason=df["reason"].str[:25])
    print(out.head(n).to_string() if n else out.to_string())

print("exact duplicate rows (all columns):", tga.duplicated(keep=False).sum(), "| extra copies:", tga.duplicated().sum())
d = tga[tga.duplicated(key + ["source"], keep=False) & tga["has_start"]].sort_values(key + ["source"])
g = d.groupby(key + ["source"])
print("key groups:", g.ngroups, "| groups with >1 distinct end date:", (g["supply_impact_end_date"].nunique(dropna=False) > 1).sum(),
      "| groups with >1 distinct rating:", (g["shortage_impact_rating"].nunique(dropna=False) > 1).sum())
show(d, 16)

print("\n--- archived: end < start or end > report date")
bad = tga[(tga["supply_impact_end_date"] < tga["supply_impact_start_date"]) |
          ((tga["source"] == "archived") & (tga["supply_impact_end_date"] > REPORT_DATE))]
show(bad)

print("\n--- archived: has end AND deletion (sample)")
show(tga[(tga["source"] == "archived") & tga["has_end"] & tga["has_deletion"]], 6)

print("\n--- active Current/Anticipated with end <= report date")
x = tga[(tga["source"] == "active") & tga["shortage_status"].isin(["Current", "Anticipated"]) & (tga["supply_impact_end_date"] <= REPORT_DATE)]
print(len(x)); show(x, 6)

exact duplicate rows (all columns): 0 | extra copies: 0
key groups: 20 | groups with >1 distinct end date: 17 | groups with >1 distinct rating: 2
     artg_id                     artg_name    source shortage_status supply_impact_start_date supply_impact_end_date deletion_from_market shortage_impact_rating                     reason
1431   11175  Bridgewest CHLORHEXIDINE 0.1  archived             NaN               2024-07-01             2026-07-31                  NaT                    Low              Manufacturing
1432   11175  Bridgewest CHLORHEXIDINE 0.1  archived             NaN               2024-07-01             2026-07-29                  NaT                    Low              Manufacturing
2108  131782  ASPEN ADRENALINE  adrenaline  archived             NaN               2019-09-25             2020-04-15                  NaT               Critical              Manufacturing
2109  131782  ASPEN ADRENALINE  adrenaline  archived             NaN               2019-09-25         

In [3]:
import numpy as np

tga["end_lt_start"] = tga["supply_impact_end_date"] < tga["supply_impact_start_date"]
tga["end_after_report"] = tga["supply_impact_end_date"] > REPORT_DATE

both = tga[tga["has_end"] & tga["has_deletion"]]
print("rows with end AND deletion:", len(both), "| deletion < end:", (both["deletion_from_market"] < both["supply_impact_end_date"]).sum())

tga["episode_type"] = np.select(
    [tga["has_deletion"], tga["shortage_status"].eq("Current"), tga["shortage_status"].eq("Anticipated"), tga["has_end"]],
    ["discontinuation", "current", "anticipated", "resolved"], default="unknown")

n0 = len(tga)
tga = tga.sort_values(["artg_id", "supply_impact_start_date", "episode_type", "supply_impact_end_date", "deletion_from_market"])
dup = tga["has_start"] & tga.duplicated(["artg_id", "supply_impact_start_date", "episode_type"], keep="last")
print("rows before dedup:", n0, "| superseded duplicates removed:", dup.sum())
print(tga[dup].groupby(["source", "episode_type"]).size().to_string())
tga = tga[~dup].reset_index(drop=True)

tga["end_passed"] = tga["episode_type"].isin(["current", "anticipated"]) & (tga["supply_impact_end_date"] <= REPORT_DATE)
tga["no_ingredient"] = tga["active_ingredients"].isna()
tga["valid_duration"] = (tga["episode_type"] == "resolved") & ~tga["end_lt_start"] & ~tga["end_after_report"]
tga["duration_days"] = np.where(tga["valid_duration"], (tga["supply_impact_end_date"] - tga["supply_impact_start_date"]).dt.days, np.nan)

print("\nrows by source x episode_type:\n", tga.groupby(["source", "episode_type"]).size().to_string())
print("\nflags by episode_type:\n", tga.groupby("episode_type")[["end_lt_start", "end_after_report", "end_passed", "no_ingredient", "valid_duration"]].sum().to_string())
print("\nvalid_duration by rating:\n", tga[tga["valid_duration"]].groupby("shortage_impact_rating").size().to_string())

PROC.mkdir(parents=True, exist_ok=True)
tga.to_csv(PROC / "tga_episodes.csv", index=False)
print("\nsaved tga_episodes.csv, shape:", tga.shape)

rows with end AND deletion: 138 | deletion < end: 100
rows before dedup: 10741 | superseded duplicates removed: 19
source    episode_type   
archived  discontinuation     3
          resolved           16

rows by source x episode_type:
 source    episode_type   
active    anticipated          91
          current             448
          discontinuation     293
          resolved            212
archived  discontinuation    1591
          resolved           8086
          unknown               1

flags by episode_type:
                  end_lt_start  end_after_report  end_passed  no_ingredient  valid_duration
episode_type                                                                              
anticipated                 0                91           0              0               0
current                     0               437          11              0               0
discontinuation             4                 4           0              0               0
resolved          

In [4]:
k = ["MONTH_OF_SUPPLY", "ITEM_CODE", "DRUG_TYPE", "PATIENT_CAT", "PHRMCY_TYPE", "SCRIPT_TYPE"]
dt = {"MONTH_OF_SUPPLY": str, "ITEM_CODE": str}
dos = pd.concat([pd.read_csv(RAW / "pbs_dos_2024-07_2025-06.csv", dtype=dt),
                 pd.read_csv(RAW / "pbs_dos_2025-07_2026-06.csv", dtype=dt)], ignore_index=True)
imap = pd.read_csv(RAW / "pbs_item_drug_map.csv", dtype=str, encoding="latin-1")
imap["drug_name"] = imap["DRUG_NAME"].str.upper().str.replace(r"\s+", " ", regex=True).str.strip()
print("DOS rows:", len(dos), "| months:", dos["MONTH_OF_SUPPLY"].nunique(), "| duplicate keys:", dos.duplicated(k).sum())
print("map duplicate ITEM_CODE:", imap["ITEM_CODE"].duplicated().sum(), "| names changed by normalisation:", (imap["drug_name"] != imap["DRUG_NAME"]).sum())

dos = dos.merge(imap[["ITEM_CODE", "drug_name", "ATC5_Code"]], on="ITEM_CODE", how="left")
um = dos["drug_name"].isna()
print("rows without drug name:", um.sum(), "| prescriptions:", dos.loc[um, "PRESCRIPTIONS"].sum(), "| items:", dos.loc[um, "ITEM_CODE"].unique()[:10])

print("\nprescriptions by PHRMCY_TYPE:\n", dos.groupby("PHRMCY_TYPE")["PRESCRIPTIONS"].sum().to_string())
pub = dos[dos["PHRMCY_TYPE"] == "S94 PUB"].copy()
print("\nS94 PUB rows:", len(pub), "| drug types:", pub.groupby("DRUG_TYPE")["PRESCRIPTIONS"].sum().to_dict())
print("\nS94 PUB prescriptions by month:\n", pub.groupby("MONTH_OF_SUPPLY")["PRESCRIPTIONS"].sum().to_string())

excl = pub["MONTH_OF_SUPPLY"].isin(["202605", "202606"])
print("\nrows dropped for months 202605-202606:", excl.sum(), "| prescriptions:", pub.loc[excl, "PRESCRIPTIONS"].sum())
pub = pub[~excl & pub["drug_name"].notna()]

top = pub.groupby("drug_name")["PRESCRIPTIONS"].sum().sort_values(ascending=False)
print("\ndistinct drug_name:", len(top)); print(top.head(15).to_string())
print("\nATC5 starts with Z (non-drug?) :", imap[imap["ATC5_Code"].str.startswith("Z", na=False)]["drug_name"].value_counts().head(12).to_dict())

pbs_pub = (pub.groupby(["MONTH_OF_SUPPLY", "drug_name"], as_index=False)["PRESCRIPTIONS"].sum()
           .rename(columns={"MONTH_OF_SUPPLY": "month", "PRESCRIPTIONS": "prescriptions"}))
pbs_pub["month"] = pd.to_datetime(pbs_pub["month"], format="%Y%m")
pbs_pub.to_csv(PROC / "pbs_public_monthly.csv", index=False)
print("\nsaved pbs_public_monthly.csv, shape:", pbs_pub.shape, "| months:", pbs_pub["month"].min(), "->", pbs_pub["month"].max())

DOS rows: 788898 | months: 24 | duplicate keys: 0
map duplicate ITEM_CODE: 0 | names changed by normalisation: 445
rows without drug name: 306 | prescriptions: 1833769 | items: <StringArray>
['00000B', '99999Z']
Length: 2, dtype: str

prescriptions by PHRMCY_TYPE:
 PHRMCY_TYPE
S90        658170406
S92            36807
S94 PRI      2788421
S94 PUB      3656321

S94 PUB rows: 164292 | drug types: {'DT': 1933, 'EP': 2003, 'GE': 3611837, 'OT': 52, 'PL': 39106, 'R1': 1241, 'RN': 97, 'UK': 52}

S94 PUB prescriptions by month:
 MONTH_OF_SUPPLY
202407    164549
202408    159039
202409    148577
202410    165231
202411    151627
202412    157881
202501    148674
202502    137979
202503    150225
202504    147509
202505    156466
202506    148966
202507    165526
202508    154688
202509    157655
202510    166350
202511    145785
202512    164631
202601    144627
202602    137149
202603    150901
202604    141576
202605    143966
202606    146744

rows dropped for months 202605-202606: 14080 | p

In [5]:
import re
SALT = set("""HYDROCHLORIDE DIHYDROCHLORIDE HYDROBROMIDE HYDRATE MONOHYDRATE DIHYDRATE TRIHYDRATE HEMIHYDRATE SESQUIHYDRATE ANHYDROUS
SODIUM POTASSIUM CALCIUM MAGNESIUM SULFATE SULPHATE PHOSPHATE ACETATE CITRATE TARTRATE BITARTRATE MALEATE FUMARATE SUCCINATE
MESILATE BESILATE TOSILATE MESYLATE BESYLATE MALATE LACTATE GLUCONATE BROMIDE NITRATE VALERATE PROPIONATE DIPROPIONATE FUROATE""".split())

def norm_comp(s):
    s = re.sub(r"\([^)]*\)", " ", str(s).upper())
    t = re.sub(r"[^A-Z0-9]+", " ", s).split()
    while len(t) > 1 and t[-1] in SALT:
        t.pop()
    return " ".join(t)

def key(s, sep):
    return " + ".join(sorted(set(norm_comp(c) for c in s.split(sep) if c.strip())))

zg = imap.groupby("drug_name")["ATC5_Code"].apply(lambda s: s.str.startswith("Z").all())
zset = set(zg[zg].index)
u = dos[dos["drug_name"].isna() & (dos["PHRMCY_TYPE"] == "S94 PUB")]
print("unmapped items in S94 PUB: rows", len(u), "| prescriptions", u["PRESCRIPTIONS"].sum(), u.groupby("ITEM_CODE")["PRESCRIPTIONS"].sum().to_dict())

pbs = pd.read_csv(PROC / "pbs_public_monthly.csv", parse_dates=["month"])
isz = pbs["drug_name"].isin(zset)
print("Z-only names:", pbs.loc[isz, "drug_name"].nunique(), "| rows:", isz.sum(), "| prescriptions:", pbs.loc[isz, "prescriptions"].sum(), "of", pbs["prescriptions"].sum())
print(pbs[isz].groupby("drug_name")["prescriptions"].sum().sort_values(ascending=False).head(6).to_string())
pbs = pbs[~isz].copy()
pbs["key"] = pbs["drug_name"].map(lambda s: key(s, "+"))
coll = pbs.groupby("key")["drug_name"].nunique()
print("\nkeys merging >1 PBS drug_name:", (coll > 1).sum())
print(pbs[pbs["key"].isin(coll[coll > 1].index)][["key", "drug_name"]].drop_duplicates().sort_values("key").head(20).to_string())

map_keys = {key(n, "+") for n in imap["drug_name"].dropna().unique()}
pub_keys = set(pbs["key"])
exact_names = set(imap["drug_name"].dropna())

t = tga[~tga["no_ingredient"]].copy()
t["ing_upper"] = t["active_ingredients"].str.upper().str.strip()
t["key"] = t["active_ingredients"].map(lambda s: key(s, "~"))
t["m_exact"] = t["ing_upper"].isin(exact_names)
t["m_map"] = t["key"].isin(map_keys)
t["m_pub"] = t["key"].isin(pub_keys)
for lab, d in [("distinct strings", t.drop_duplicates("ing_upper")), ("episodes", t)]:
    n = len(d)
    print(f"\n{lab}: {n} | exact: {d.m_exact.sum()} ({d.m_exact.mean():.1%}) | key in PBS map: {d.m_map.sum()} ({d.m_map.mean():.1%}) | key in S94 PUB usage: {d.m_pub.sum()} ({d.m_pub.mean():.1%})")
print("distinct keys: TGA", t["key"].nunique(), "| matched to PBS map", t.loc[t.m_map, "key"].nunique(), "| matched to S94 PUB", t.loc[t.m_pub, "key"].nunique())
print("\ntop unmatched TGA strings by episodes:")
print(t[~t["m_map"]].groupby("ing_upper").size().sort_values(ascending=False).head(30).to_string())


unmapped items in S94 PUB: rows 80 | prescriptions 149 {'00000B': 52, '99999Z': 97}
Z-only names: 1 | rows: 22 | prescriptions: 1908 of 3365474
drug_name
EXTEMPORANEOUSLY PREPARED    1908

keys merging >1 PBS drug_name: 6
                 key                                               drug_name
75     BETAMETHASONE                              BETAMETHASONE DIPROPIONATE
76     BETAMETHASONE                                  BETAMETHASONE VALERATE
713    BETAMETHASONE  BETAMETHASONE ACETATE + BETAMETHASONE SODIUM PHOSPHATE
236          FERROUS                                        FERROUS FUMARATE
238          FERROUS                                         FERROUS SULFATE
251      FLUTICASONE                                  FLUTICASONE PROPIONATE
2820     FLUTICASONE                                     FLUTICASONE FUROATE
296   HYDROCORTISONE                                          HYDROCORTISONE
297   HYDROCORTISONE                                  HYDROCORTISONE ACETATE
298   HY

In [6]:
names = pd.Series(sorted(imap["drug_name"].dropna().unique()))
um = (t[~t["m_map"]].groupby("key").agg(episodes=("key", "size"), example=("ing_upper", "first"))
      .sort_values("episodes", ascending=False))

def cands(kx):
    tok = kx.split(" + ")[0].split()[0]
    return names[names.str.contains(r"\b" + re.escape(tok) + r"\b", regex=True)].tolist()

um["cand"] = [cands(kx) for kx in um.index]
um["n_cand"] = um["cand"].str.len()
print("unmatched keys:", len(um), "| episodes:", um["episodes"].sum())
print("no PBS name contains first token of key: keys", (um.n_cand == 0).sum(), "| episodes", um.loc[um.n_cand == 0, "episodes"].sum())
print("has candidates but no key match: keys", (um.n_cand > 0).sum(), "| episodes", um.loc[um.n_cand > 0, "episodes"].sum())
out = um.head(60).assign(example=um.head(60)["example"].str[:42],
                         pbs_candidates=um.head(60)["cand"].map(lambda c: "; ".join(c[:3])[:90]))
print(out[["episodes", "example", "n_cand", "pbs_candidates"]].to_string(index=False))

unmatched keys: 395 | episodes: 2423
no PBS name contains first token of key: keys 238 | episodes 1174
has candidates but no key match: keys 157 | episodes 1249
 episodes                                    example  n_cand                                                                             pbs_candidates
       70             ROSUVASTATIN CALCIUM~EZETIMIBE       5                          ATORVASTATIN (&) EZETIMIBE; EZETIMIBE; EZETIMIBE (&) ROSUVASTATIN
       62   AMLODIPINE BESILATE~OLMESARTAN MEDOXOMIL       8                              AMLODIPINE; AMLODIPINE + ATORVASTATIN; AMLODIPINE + VALSARTAN
       59                      CANDESARTAN CILEXETIL       2                                             CANDESARTAN; CANDESARTAN + HYDROCHLOROTHIAZIDE
       59                  ROPIVACAINE HYDROCHLORIDE       0                                                                                           
       53   OLMESARTAN MEDOXOMIL~HYDROCHLOROTHIAZIDE      14 AMILORIDE + HYDROC

In [7]:
SALT2 = SALT | set("""CILEXETIL MEDOXOMIL ERBUMINE OXALATE MOFETIL BENZOATE ARGININE HYDROGEN DISODIUM TROMETAMOL HYCLATE CHLORIDE
DIMESILATE HEMIHEPTAHYDRATE PENTAHYDRATE ETHYL ETHYLSUCCINATE LACTOBIONATE""".split())
SYN = {"CLAVULANATE": "CLAVULANIC ACID"}

def comp2(c):
    c = re.sub(r"\([^)]*\)", " ", c.upper())
    c = re.sub(r"\bACID\s+(HYDROGEN\s+)?TARTRATE\b", " ", c)
    tok = re.sub(r"[^A-Z0-9]+", " ", c).split()
    core = [x for x in tok if x not in SALT2] or tok
    s = " ".join(core)
    return SYN.get(s, s)

def key2(s):
    s = str(s).upper().replace("(&)", "+").replace("~", "+")
    return " + ".join(sorted({comp2(c) for c in s.split("+") if comp2(c)}))

pbs["key2"] = pbs["drug_name"].map(key2)
imap["key2"] = imap["drug_name"].map(key2)
map_keys2, pub_keys2 = set(imap["key2"]), set(pbs["key2"])
coll = pbs.groupby("key2")["drug_name"].nunique()
print("PBS keys merging >1 drug_name:", (coll > 1).sum(), "(v1: 6)")
print(pbs[pbs["key2"].isin(coll[coll > 1].index)][["key2", "drug_name"]].drop_duplicates().sort_values("key2").head(30).to_string(index=False))

t["key2"] = t["active_ingredients"].map(key2)
t["m_map2"], t["m_pub2"] = t["key2"].isin(map_keys2), t["key2"].isin(pub_keys2)
rate = lambda d, c: f"{d[c].sum()} ({d[c].mean():.1%})"
for lab, d in [("distinct strings", t.drop_duplicates("ing_upper")), ("episodes", t)]:
    print(f"\n{lab}: {len(d)} | map v1 {rate(d,'m_map')} -> v2 {rate(d,'m_map2')} | S94 PUB v1 {rate(d,'m_pub')} -> v2 {rate(d,'m_pub2')}")
print("distinct keys v2: TGA", t["key2"].nunique(), "| in map", t.loc[t.m_map2, "key2"].nunique(), "| in S94 PUB", t.loc[t.m_pub2, "key2"].nunique())
print("episodes in map but no S94 PUB usage in window:", (t.m_map2 & ~t.m_pub2).sum())

pk = pd.Series(sorted(map_keys2))
def cands2(kx):
    tok = kx.split(" + ")[0].split()[0]
    return pk[pk.str.contains(r"\b" + re.escape(tok) + r"\b", regex=True)].tolist()
um2 = (t[~t.m_map2].groupby("key2").agg(episodes=("key2", "size"), example=("ing_upper", "first"))
       .sort_values("episodes", ascending=False))
um2["cand"] = [cands2(kx) for kx in um2.index]
um2["n_cand"] = um2["cand"].str.len()
print("\nunmatched keys:", len(um2), "| episodes:", um2["episodes"].sum())
print("no candidate: keys", (um2.n_cand == 0).sum(), "| episodes", um2.loc[um2.n_cand == 0, "episodes"].sum())
print("with candidates: keys", (um2.n_cand > 0).sum(), "| episodes", um2.loc[um2.n_cand > 0, "episodes"].sum())
fix = um2[um2.n_cand > 0].head(40)
print(fix.assign(key2=fix.index.str[:45], pbs_candidates=fix["cand"].map(lambda c: "; ".join(c[:3])[:90]))
      [["episodes", "key2", "n_cand", "pbs_candidates"]].to_string(index=False))

PBS keys merging >1 drug_name: 8 (v1: 6)
                      key2                                              drug_name
             BETAMETHASONE                             BETAMETHASONE DIPROPIONATE
             BETAMETHASONE                                 BETAMETHASONE VALERATE
             BETAMETHASONE BETAMETHASONE ACETATE + BETAMETHASONE SODIUM PHOSPHATE
              ERYTHROMYCIN                                           ERYTHROMYCIN
              ERYTHROMYCIN                            ERYTHROMYCIN ETHYLSUCCINATE
ESTRADIOL + NORETHISTERONE                     ESTRADIOL + NORETHISTERONE ACETATE
ESTRADIOL + NORETHISTERONE       ESTRADIOL (&) ESTRADIOL + NORETHISTERONE ACETATE
                   FERROUS                                       FERROUS FUMARATE
                   FERROUS                                        FERROUS SULFATE
               FLUTICASONE                                    FLUTICASONE FUROATE
               FLUTICASONE                               

In [8]:
tga["ingredient_key"] = tga["active_ingredients"].map(lambda s: key2(s) if pd.notna(s) else np.nan)
tga["in_pbs_map"] = tga["ingredient_key"].isin(map_keys2)
tga["in_pbs_public"] = tga["ingredient_key"].isin(pub_keys2)
out = tga.drop(columns=["has_start", "has_end", "has_deletion", "phone"], errors="ignore")
out.to_csv(PROC / "tga_episodes.csv", index=False)

pbs_key = (pbs.groupby(["month", "key2"], as_index=False)["prescriptions"].sum()
           .rename(columns={"key2": "ingredient_key"}))
lookup = (pbs.groupby("key2")["drug_name"].apply(lambda s: " | ".join(sorted(set(s)))).reset_index()
          .rename(columns={"key2": "ingredient_key", "drug_name": "pbs_drug_names"}))
assert pbs_key["prescriptions"].sum() == pbs["prescriptions"].sum()
pbs_key.to_csv(PROC / "pbs_public_monthly.csv", index=False)
lookup.to_csv(PROC / "pbs_ingredient_lookup.csv", index=False)

print("tga_episodes", out.shape, "| pbs_public_monthly", pbs_key.shape, "| pbs_ingredient_lookup", lookup.shape)
print("months:", pbs_key["month"].min(), "->", pbs_key["month"].max(), "| distinct keys:", pbs_key["ingredient_key"].nunique())
print("TGA episodes with key:", out["ingredient_key"].notna().sum(), "| in map:", out["in_pbs_map"].sum(), "| with S94 PUB usage:", out["in_pbs_public"].sum())

tga_episodes (10722, 26) | pbs_public_monthly (14142, 3) | pbs_ingredient_lookup (742, 2)
months: 2024-07-01 00:00:00 -> 2026-04-01 00:00:00 | distinct keys: 742
TGA episodes with key: 10721 | in map: 9116 | with S94 PUB usage: 7915
